Assignment:
Representation Learning using PCA and Autoencoders

Datasets:
1. MNIST
2. CIFAR10 (converted to grayscale)

Image Size:
28 x 28

Intensity Range:
50 - 200

Dataset Split:
70% Train
20% Validation
10% Test

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import mnist, cifar10

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

Load & Preprocess Data :

Convert CIFAR10 to grayscale
Resize both datasets to 28x28
Normalize intensity values between 50 and 200

In [ ]:
# Function for Intensity Normalizzation
def normalize_50_200(images):
    """
    Converts image intensities to range [50,200]

    Formula:
    new_pixel = 50 + 150 * (pixel-min)/(max-min)
    """

    images = images.astype(np.float32)

    mn = images.min()
    mx = images.max()

    images = 50 + 150 * (images - mn) / (mx - mn)

    return images

# Loading CIFAR
# CIFAR10 images are 32x32 RGB

(x_cifar, y_cifar), _ = cifar10.load_data()

y_cifar = y_cifar.flatten()

print(x_cifar.shape)

#Gray = 0.299*R + 0.587*G + 0.114*B
#Standard luminance formula

x_cifar_gray = (
    0.299 * x_cifar[:,:,:,0]
    + 0.587 * x_cifar[:,:,:,1]
    + 0.114 * x_cifar[:,:,:,2]
)

x_cifar_gray = np.expand_dims(
    x_cifar_gray,
    axis=-1
)

#Resize CIFAR10*  to 28 * 28
x_cifar = tf.image.resize(
    x_cifar_gray,
    (28,28)
).numpy()

#Loading MNIST
(x_mnist, y_mnist), _ = mnist.load_data()

x_mnist = np.expand_dims(
    x_mnist,
    axis=-1
)

x_mnist = tf.image.resize(
    x_mnist,
    (28,28)
).numpy()

#Normalizing Intensities
x_mnist = normalize_50_200(x_mnist)
x_cifar = normalize_50_200(x_cifar)

Train / Validation / Test Split
Requirement

70% Training

20% Validation

10% Test

In [ ]:
def create_split(x, y):

    """
    First split:
    70% train
    30% temporary
    """

    x_train, x_temp, y_train, y_temp = train_test_split(
        x,
        y,
        test_size=0.30,
        stratify=y,
        random_state=42
    )

    """
    Split remaining 30%

    Validation = 20%
    Test = 10%

    Therefore:
    20/30 = 2/3 validation
    10/30 = 1/3 test
    """

    x_val, x_test, y_val, y_test = train_test_split(
        x_temp,
        y_temp,
        test_size=1/3,
        stratify=y_temp,
        random_state=42
    )

    return (
        x_train,
        y_train,
        x_val,
        y_val,
        x_test,
        y_test
    )

mnist_data = create_split(
    x_mnist,
    y_mnist
)

cifar_data = create_split(
    x_cifar,
    y_cifar
)